# Baseline Model - Random Forest Regressor

## 1. Load data

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

print("Loading training and testing datasets...")
df_train = pd.read_csv("data/seoul_2021_train.csv")
df_test = pd.read_csv("data/seoul_2021_test.csv")

# Prepare Training Data (X_train and y_train)
X_train = df_train.drop(columns=['pm2.5'])
y_train = df_train['pm2.5']

# Prepare Testing Data (X_test and y_test)
X_test = df_test.drop(columns=['pm2.5'])
y_test = df_test['pm2.5']

print(f"Ready! Training features: {X_train.shape}, Testing features: {X_test.shape}")

Loading training and testing datasets...
Ready! Training features: (167023, 5), Testing features: (41756, 5)


## 2. Build Model 

In [3]:
print("Initializing the Random Forest Baseline Model...")

# 100 decision trees, using all CPU cores (n_jobs=-1) for speed
baseline_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

print("Model built successfully!")

Initializing the Random Forest Baseline Model...
Model built successfully!


## 3. Train Model

In [4]:
print("Training the model.")

baseline_model.fit(X_train, y_train)

print("Training complete.")

Training the model.
Training complete.


## 4. Evaluate Model

In [5]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error

print("Evaluating Model Performance on test data...")
predictions = baseline_model.predict(X_test)

# Calculate standard error metrics (Prof's requirements)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

# Calculate Percentage Error (Dein Prozent-Wunsch!)
mape = mean_absolute_percentage_error(y_test, predictions) * 100

print("\n Model Evaluation")
print(f"MSE: {mse:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE (Average Error in %): {mape:.2f}%")

# Show a quick comparison of the first 5 values
print("\nFirst 5 Predictions vs. Actuals")
print(f"Predicted PM2.5: {predictions[:5].round(2)}")
print(f"Actual PM2.5:    {y_test[:5].values}")

Evaluating Model Performance on test data...

 Model Evaluation
MSE: 40.48
R2 Score: 0.8671
MAPE (Average Error in %): 34.09%

First 5 Predictions vs. Actuals
Predicted PM2.5: [ 9.94 17.3  70.61 21.08  7.27]
Actual PM2.5:    [ 9. 22. 81. 22.  7.]


### Model Robustness: 5-Fold Cross-Validation
To ensure our Random Forest baseline model actually learns the underlying patterns and doesn't just memorize the training data (overfitting), we are performing a 5-Fold Cross-Validation. 

This technique splits the training data into 5 separate parts (folds), training and testing the model multiple times. By averaging the results, we get a highly reliable confirmation of our model's true performance metrics (R2, MSE, and MAPE).

In [7]:
from sklearn.model_selection import cross_val_score, KFold

# Define KFold parameters (split data into 5 random folds)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Starting 5-Fold Cross-Validation")

# Calculate the main metrics across all folds
cv_r2_scores = cross_val_score(baseline_model, X_train, y_train, cv=kf, scoring='r2')
cv_mse_scores = -cross_val_score(baseline_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
cv_mape_scores = -cross_val_score(baseline_model, X_train, y_train, cv=kf, scoring='neg_mean_absolute_percentage_error') * 100

# Print the formatted average results for the performance report
print(f"Average R2 Score: {cv_r2_scores.mean():.4f}")
print(f"Average MSE:      {cv_mse_scores.mean():.4f}")
print(f"Average MAPE:     {cv_mape_scores.mean():.2f}%")

Starting 5-Fold Cross-Validation
Average R2 Score: 0.8707
Average MSE:      40.1001
Average MAPE:     34.07%


### Log the final scores 

In [8]:
# Save metrics to a text file for the performance report
with open("model_metrics_report.txt", "w") as file:
    file.write("Random Forest Baseline Metrics (5-Fold CV)\n")
    file.write(f"Average R2 Score: {cv_r2_scores.mean():.4f}\n")
    file.write(f"Average MSE:      {cv_mse_scores.mean():.4f}\n")
    file.write(f"Average MAPE:     {cv_mape_scores.mean():.2f}%\n")
    file.write("Status: Model is robust and validated.\n")

print("Metrics successfully saved to 'model_metrics_report.txt'!")

Metrics successfully saved to 'model_metrics_report.txt'!
